# Redes Neurais Artificiais: do problema ao neurônio

## Objetivo

Este notebook apresenta, passo a passo, o que é um neurônio artificial e uma rede neural, partindo de um problema que os métodos já estudados (Regressão Linear e Regressão Polinomial) não resolvem de forma satisfatória. A terminologia técnica só é introduzida depois que o mecanismo foi construído e observado.

## Como utilizar

- Execute as células na ordem em que aparecem.
- Cada célula de código é precedida de uma explicação do que ela faz e seguida de uma interpretação do resultado.
- As células marcadas com `COMPLETE` contêm lacunas (`____`) que devem ser preenchidas antes da execução.
- Duração estimada: cerca de 2 horas.

## Roteiro

1. Definição do problema
2. Primeira abordagem: Regressão Linear
3. Segunda abordagem: Regressão Polinomial
4. Uma abordagem alternativa: rampas
5. Soma de rampas
6. Ajuste automático dos valores
7. Terminologia: neurônio, camada e pesos
8. Implementação com o Scikit-Learn
9. Síntese

## Passo 1 — Definição do problema

Considere um experimento culinário: um bolo é assado por tempos diferentes e recebe uma nota de 0 a 10.

- tempo curto: a massa fica crua e a nota é baixa;
- tempo adequado: a nota é alta;
- tempo excessivo: o bolo queima e a nota volta a ser baixa.

O objetivo é construir um modelo que **preveja a nota a partir do tempo no forno**. Como o valor a ser previsto é numérico, trata-se de um problema de **regressão**.

A célula abaixo (a) importa as bibliotecas NumPy (cálculo numérico) e Matplotlib (gráficos), (b) gera 90 observações do experimento e (c) as exibe em um gráfico de dispersão. As três linhas que geram os dados apenas simulam o experimento; não é necessário compreendê-las.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Gerando os dados do experimento (nao precisa entender estas 3 linhas)
rng = np.random.RandomState(0)
tempo = np.sort(rng.uniform(0, 60, 90))
nota = np.interp(tempo, [0, 15, 30, 40, 55, 60], [0, 0, 10, 10, 0, 0]) + rng.normal(0, 0.5, 90)

plt.scatter(tempo, nota, color="gray", s=18)
plt.xlabel("minutos no forno")
plt.ylabel("nota do bolo")
plt.show()

**Interpretação.** Os pontos formam uma montanha com topo achatado: a nota é baixa até cerca de 15 minutos, sobe, permanece alta entre cerca de 30 e 40 minutos e volta a cair. Nenhuma reta descreve esse comportamento, pois uma reta só consegue subir ou só consegue descer.

## Passo 2 — Primeira abordagem: Regressão Linear

A Regressão Linear ajusta aos dados a reta que minimiza o erro. Para medir a qualidade do ajuste utiliza-se o **MSE** (erro quadrático médio): a média dos quadrados das diferenças entre o valor previsto e o valor real. Quanto menor o MSE, melhor o ajuste.

Na célula abaixo:

1. `X = tempo.reshape(-1, 1)` converte a entrada em uma tabela de uma coluna, formato exigido pelo Scikit-Learn;
2. `LinearRegression().fit(X, nota)` treina o modelo. **COMPLETE** os argumentos de `fit` (entrada e alvo);
3. `predict` calcula a nota prevista para cada tempo;
4. o MSE é calculado e exibido no título do gráfico.

In [ ]:
from sklearn.linear_model import LinearRegression

X = tempo.reshape(-1, 1)    # o sklearn quer a entrada como tabela (1 coluna)

# COMPLETE: treine a regressao linear (entrada X, alvo nota)
modelo_reta = LinearRegression().fit(X, nota)

previsao_reta = modelo_reta.predict(X)
erro_reta = np.mean((previsao_reta - nota) ** 2)

plt.scatter(tempo, nota, color="lightgray", s=18)
plt.plot(tempo, previsao_reta, color="#B0413E", linewidth=3)
plt.title(f"Regressao Linear | erro (MSE) = {erro_reta:.2f}")
plt.xlabel("minutos no forno"); plt.ylabel("nota do bolo")
plt.show()

**Interpretação.** O erro é elevado (cerca de 16). A reta apresenta uma única tendência, crescente ou decrescente, e por isso prevê que um tempo maior de forno é sempre melhor ou sempre pior, ignorando a existência de um tempo ideal.

## Passo 3 — Segunda abordagem: Regressão Polinomial

A Regressão Polinomial permite curvar a reta ao incluir potências do tempo (t², t³, ...). O parâmetro *grau* define a maior potência utilizada. Serão avaliados o grau 2 e o grau 6.

A célula abaixo define três funções auxiliares:

- `ajustar_polinomial(grau)`: cria as potências do tempo e treina o modelo (o tempo é dividido por 10 apenas para evitar números muito grandes);
- `prever_polinomial(modelo, pf, minutos)`: calcula a previsão do modelo para tempos quaisquer;
- `mostrar_polinomial(grau, ate)`: desenha os dados e a curva ajustada e informa o MSE.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

def ajustar_polinomial(grau):
    pf = PolynomialFeatures(degree=grau, include_bias=False)
    X_dez = (tempo / 10).reshape(-1, 1)   # tempo em dezenas de minutos (so pra os numeros nao ficarem gigantes)
    modelo = LinearRegression().fit(pf.fit_transform(X_dez), nota)
    return modelo, pf

def prever_polinomial(modelo, pf, minutos):
    return modelo.predict(pf.transform((minutos / 10).reshape(-1, 1)))

def mostrar_polinomial(grau, ate=60):
    modelo, pf = ajustar_polinomial(grau)
    grade = np.linspace(0, ate, 300)
    erro = np.mean((prever_polinomial(modelo, pf, tempo) - nota) ** 2)
    plt.figure(figsize=(8, 4.5))
    plt.scatter(tempo, nota, color="lightgray", s=18)
    plt.plot(grade, prever_polinomial(modelo, pf, grade), color="#2F6DB5", linewidth=3)
    if ate > 60:
        plt.axvline(60, color="gray", linestyle="--")
        plt.text(60.5, 20, "fim dos dados", color="gray")
        plt.ylim(-25, 30)
    else:
        plt.ylim(-6, 14)
    plt.title(f"Polinomial grau {grau} | erro (MSE) = {erro:.2f}")
    plt.xlabel("minutos no forno"); plt.ylabel("nota do bolo")
    plt.show()
    return modelo, pf

A célula seguinte executa o ajuste com os graus 2 e 6.

In [ ]:
mostrar_polinomial(2)
mostrar_polinomial(6)

**Interpretação.** O grau 2 (MSE ≈ 3,9) não acompanha o topo achatado da montanha. O grau 6 (MSE ≈ 0,3) acompanha bem os dados. Dentro da faixa observada, portanto, a Regressão Polinomial é uma solução adequada para este problema, que tem uma única entrada. Duas limitações justificam, no entanto, a busca de outra abordagem.

**Limitação 1: previsões fora da faixa dos dados.** Os dados cobrem de 0 a 60 minutos. A célula abaixo estende o gráfico até 80 minutos e imprime a nota prevista para esse tempo.

In [ ]:
modelo6, pf6 = mostrar_polinomial(6, ate=80)
modelo2, pf2 = mostrar_polinomial(2, ate=80)
print("Nota prevista para 80 min -> grau 6:", round(prever_polinomial(modelo6, pf6, np.array([80.0]))[0], 1))
print("Nota prevista para 80 min -> grau 2:", round(prever_polinomial(modelo2, pf2, np.array([80.0]))[0], 1))

**Interpretação.** Para 80 minutos, o grau 6 prevê cerca de 24,7 e o grau 2 prevê cerca de -19,9, valores impossíveis em uma escala de 0 a 10. Fora da faixa observada, o polinômio diverge.

**Limitação 2: número de colunas com várias entradas.** Na prática costuma haver mais de uma variável de entrada. A célula abaixo mostra quantas colunas a Regressão Polinomial de grau 3 precisa criar conforme aumenta o número de entradas.

In [ ]:
for n_entradas in [2, 5, 10, 20]:
    n_colunas = PolynomialFeatures(degree=3, include_bias=False).fit(np.zeros((1, n_entradas))).n_output_features_
    print(f"{n_entradas:2d} entradas, grau 3  ->  {n_colunas} colunas")

**Interpretação.** O número de colunas cresce rapidamente: 10 entradas já exigem 285 colunas e 20 entradas exigem 1770. A abordagem se torna progressivamente menos prática à medida que o número de variáveis aumenta.

**Conclusão do passo.** A Regressão Polinomial resolve o problema com uma entrada dentro da faixa observada, mas diverge fora dela e escala mal com o número de entradas. O passo seguinte apresenta uma alternativa.

## Passo 4 — Uma abordagem alternativa: rampas

Em vez de curvar a reta, a alternativa consiste em **emendar pedaços retos**. O elemento básico é a **rampa**:

> o valor é zero até determinado minuto; a partir dele, cresce (ou decresce) em linha reta.

Uma rampa é definida por dois números: **onde começa** (em minutos) e **inclinação** (pontos de nota por minuto). O ponto em que a rampa deixa de ser parada e passa a ser inclinada chama-se **dobra**.

A célula abaixo define a função `rampa`, que calcula o valor de uma rampa, e a função `mostrar_rampa`, que a desenha. A última linha exibe um exemplo: rampa que começa em 20 minutos, com inclinação 0,5.

In [ ]:
def rampa(t, onde_comeca, inclinacao):
    """Fica em 0 ate 'onde_comeca'. Depois disso, sobe (ou desce) em linha reta."""
    return inclinacao * np.maximum(0, t - onde_comeca)

def mostrar_rampa(onde_comeca, inclinacao):
    grade = np.linspace(0, 60, 300)
    plt.plot(grade, rampa(grade, onde_comeca, inclinacao), color="#00A868", linewidth=3)
    plt.axvline(onde_comeca, color="gray", linestyle=":")
    plt.ylim(-10, 15)
    plt.title(f"Rampa: comeca em {onde_comeca} min, inclinacao {inclinacao}")
    plt.xlabel("minutos no forno")
    plt.show()

mostrar_rampa(20, 0.5)     # <- altere estes 2 numeros e execute novamente

**Atividade.** Altere os dois valores passados a `mostrar_rampa` (última linha da célula) e execute novamente. Efeitos esperados:

- aumentar `onde_comeca` desloca a dobra para a direita;
- aumentar o valor absoluto de `inclinacao` torna a rampa mais íngreme;
- uma `inclinacao` negativa faz a rampa descer.

## Passo 5 — Soma de rampas

A ideia central é **somar várias rampas**. Cada rampa adicionada cria uma nova dobra na curva resultante. Uma curva com n dobras é composta por n + 1 trechos retos.

A função `mostrar`, definida abaixo, recebe uma lista de rampas, desenha cada uma (tracejada) e a soma (verde) e informa o MSE. O parâmetro `nivel_base` é a altura inicial da curva.

In [ ]:
def mostrar(rampas, nivel_base=0, titulo=""):
    grade = np.linspace(0, 60, 300)
    soma = np.full_like(grade, nivel_base, dtype=float)

    plt.figure(figsize=(8, 4.5))
    plt.scatter(tempo, nota, color="lightgray", s=18, label="dados")
    for i, (onde, incl) in enumerate(rampas, start=1):
        r = rampa(grade, onde, incl)
        plt.plot(grade, r, "--", linewidth=1.5, label=f"rampa {i} (comeca em {onde:.0f} min)")
        soma += r
    plt.plot(grade, soma, color="#00A868", linewidth=3, label="soma das rampas")

    previsto = nivel_base + sum(rampa(tempo, o, i) for o, i in rampas)
    erro = np.mean((previsto - nota) ** 2)
    plt.title(f"{titulo}  |  erro (MSE) = {erro:.2f}")
    plt.xlabel("minutos no forno"); plt.ylabel("nota do bolo")
    plt.ylim(-6, 14); plt.legend(fontsize=8)
    plt.show()

**Atividade: ajuste manual de duas rampas.** Cada rampa é um par `(onde_comeca, inclinacao)`. Substitua os valores da célula seguinte para que a soma das duas rampas acompanhe a montanha (objetivo: MSE inferior a 3). Orientação:

1. Estime, no gráfico de dispersão, em que minuto a nota começa a subir (cerca de 13 a 15 minutos) e use esse valor como `onde_comeca` da primeira rampa.
2. Estime a inclinação da subida: a nota sobe cerca de 10 pontos em cerca de 20 minutos, isto é, aproximadamente 0,5 ponto por minuto.
3. A segunda rampa deve começar no pico da montanha (cerca de 35 minutos).
4. **A segunda rampa é somada à primeira**: a partir de onde ela começa, as inclinações se somam. Para que a curva desça com inclinação aproximada de -0,55, a inclinação da segunda rampa deve ser aproximadamente -0,55 - 0,5 = -1,05.

In [ ]:
# cada par = (onde comeca em minutos, inclinacao)  -> substitua os numeros
minhas_rampas = [(0, 0), (0, 0)]

mostrar(minhas_rampas, titulo="Ajuste manual")

A célula seguinte mostra o efeito de 1, 2, 3 e 4 dobras, com valores já ajustados.

In [ ]:
mostrar([(47, -0.52)], nivel_base=5.25, titulo="1 dobra")
mostrar([(13, 0.53), (35, -1.09)], titulo="2 dobras")
mostrar([(13, 0.53), (35, -1.09), (56, 0.55)], titulo="3 dobras")
mostrar([(15, 0.667), (30, -0.667), (40, -0.667), (55, 0.667)], titulo="4 dobras")

**Interpretação.**

- Com 1 dobra, o MSE é de aproximadamente 15,5, praticamente igual ao da reta (cerca de 16,1): a curva muda de direção apenas uma vez, e a montanha exige subir e depois descer.
- Com 2 dobras, o MSE cai para aproximadamente 0,7. Duas dobras é a menor quantidade que permite subir e voltar a descer.
- Com 3 e 4 dobras, o MSE cai para aproximadamente 0,6 e 0,3. Os ganhos são menores, pois o que resta é principalmente o ruído dos dados.
- Em cada gráfico, o número de trechos retos da curva verde é igual ao número de dobras mais um.

## Passo 6 — Ajuste automático dos valores

Com duas rampas ainda é possível estimar os valores manualmente; com dezenas, não. O computador pode procurar os valores por tentativa e erro, com o seguinte procedimento:

1. atribuir valores iniciais aleatórios às rampas;
2. calcular o erro (MSE);
3. alterar levemente os valores;
4. manter a alteração apenas se o erro diminuir;
5. repetir milhares de vezes.

Esse procedimento é, em essência, o **treinamento** de um modelo.

A célula abaixo define a função `buscar_rampas`. Seus parâmetros são o número de rampas, o número de passos e a `semente`, que determina os valores iniciais aleatórios. A função devolve os valores encontrados, o erro final e o histórico do erro a cada tentativa.

In [ ]:
def buscar_rampas(n_rampas, passos=20000, semente=0):
    """Busca por tentativa e erro: chuta, mede o erro, mantem a mudanca so se melhorar."""
    r = np.random.RandomState(semente)
    onde = r.uniform(0, 60, n_rampas)      # chute inicial: onde cada rampa comeca
    incl = r.normal(0, 0.5, n_rampas)      # chute inicial: inclinacao de cada rampa
    base = 0.0                             # chute inicial: nivel base

    def erro_de(o, i, b):
        previsto = b + sum(rampa(tempo, a, c) for a, c in zip(o, i))
        return np.mean((previsto - nota) ** 2)

    erro_atual = erro_de(onde, incl, base)
    historico = [erro_atual]
    tamanho_do_passo = 1.0
    for passo in range(passos):
        novo_onde = onde + r.normal(0, 3.0 * tamanho_do_passo, n_rampas)
        nova_incl = incl + r.normal(0, 0.15 * tamanho_do_passo, n_rampas)
        nova_base = base + r.normal(0, 0.5 * tamanho_do_passo)
        novo_erro = erro_de(novo_onde, nova_incl, nova_base)
        if novo_erro < erro_atual:          # so aceita se o erro CAIU
            onde, incl, base, erro_atual = novo_onde, nova_incl, nova_base, novo_erro
        historico.append(erro_atual)
        if (passo + 1) % (passos // 4) == 0:
            tamanho_do_passo *= 0.5         # vai refinando: passos menores com o tempo
    return onde, incl, base, erro_atual, historico

A célula seguinte executa a busca com 2 rampas, desenha o resultado e o histórico do erro (em escala logarítmica).

In [ ]:
onde, incl, base, erro_final, historico = buscar_rampas(2, semente=0)

mostrar(list(zip(onde, incl)), nivel_base=base, titulo="Achado pelo computador (2 rampas)")

plt.figure(figsize=(8, 4.5))
plt.plot(historico, color="#00A868")
plt.yscale("log")
plt.xlabel("tentativa"); plt.ylabel("erro (MSE, escala log)")
plt.title("O erro caindo enquanto o computador tenta")
plt.show()

**Interpretação.** Para a semente 0, a busca encontra rampas que começam em cerca de 13 e 35 minutos, valores próximos dos ajustados manualmente, com MSE de aproximadamente 0,6. O gráfico do histórico mostra que o erro inicial é muito alto (acima de 100) e cai para menos de 2 em cerca de 150 tentativas; nas tentativas seguintes a melhoria é gradual.

A célula seguinte repete a busca com 10 sementes diferentes (10 pontos de partida aleatórios), para 2 e para 3 rampas.

In [ ]:
for n_rampas in [2, 3]:
    erros = [buscar_rampas(n_rampas, semente=s)[3] for s in range(10)]
    print(f"{n_rampas} rampas:", [round(float(e), 2) for e in erros])

**Interpretação.** Com 2 rampas, 7 das 10 execuções chegam a MSE de aproximadamente 0,59, enquanto 3 execuções permanecem em torno de 15 a 16, erro equivalente ao de 1 dobra ou ao da reta. Com 3 rampas, todas as execuções chegam a valores inferiores a 1,2.

A explicação para as execuções que não convergem é que uma rampa pode ser levada a uma posição em que não influencia o erro (por exemplo, começando após os 60 minutos, onde não há dados). Como pequenas alterações não modificam o erro, a busca não tem indicação de qual direção seguir. Em redes neurais, esse fenômeno é conhecido como "ReLU morto". Quanto mais rampas, maior a chance de sucesso.

## Passo 7 — Terminologia: neurônio, camada e pesos

O mecanismo construído nos passos anteriores corresponde a uma **rede neural artificial**. Cada rampa é uma **reta** (z = w·t + b, a mesma expressão da Regressão Linear) seguida de uma **dobra**: a função ReLU, que substitui valores negativos por zero. Essa combinação de reta e dobra é um **neurônio artificial**.

A célula abaixo (a) define a função `relu` e a função `neuronio`, (b) desenha a reta z (pontilhada) e o resultado após a ReLU (verde) e (c) verifica numericamente que a rampa com início em 20 minutos e inclinação 0,5 é igual a um neurônio com w = 1 e b = -20 multiplicado por um peso de saída 0,5.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def neuronio(t, w, b):
    z = w * t + b        # <- uma reta! (a mesma conta da Regressao Linear)
    return relu(z)       # <- a "dobra"

t = np.linspace(0, 60, 300)
plt.plot(t, 1 * t - 20, ":", color="gray", label="z = w*t + b   (a reta)")
plt.plot(t, neuronio(t, w=1, b=-20), color="#00A868", linewidth=3, label="ReLU(z)   (a reta com a dobra)")
plt.ylim(-25, 40); plt.xlabel("minutos no forno"); plt.legend()
plt.title("Um neuronio")
plt.show()

# prova: nossa rampa (onde=20, inclinacao=0.5) e 1 neuronio multiplicado por um peso de saida (0.5)
print("rampa == neuronio?", np.allclose(rampa(t, 20, 0.5), 0.5 * neuronio(t, w=1, b=-20)))

**Interpretação.** O gráfico mostra que a ReLU preserva a reta onde ela é positiva e a "corta" em zero onde é negativa, o que produz a dobra. A última linha imprime `True`, confirmando que uma rampa e um neurônio são equivalentes.

**Nota sobre os parâmetros.** Na rede neural, cada neurônio guarda `w` e `b`, e a saída guarda um peso `v` por neurônio mais um nível base `c`. Os dois números de uma rampa são combinações desses valores (`onde = -b/w` e `inclinação = v·w`). Além disso, na rede a rampa também pode ficar virada para o outro lado, isto é, ativar-se *antes* de certo minuto em vez de depois.

A rede completa calcula:

**nota prevista = c + v₁·ReLU(w₁·t + b₁) + v₂·ReLU(w₂·t + b₂) + v₃·ReLU(w₃·t + b₃)**

A célula seguinte desenha essa estrutura para 3 neurônios.

In [ ]:
def desenhar_rede(n_neuronios):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.axis("off")
    ys = list(np.linspace(0.8, 0.2, n_neuronios)) if n_neuronios > 1 else [0.5]
    for y in ys:
        ax.plot([0.1, 0.5], [0.5, y], color="lightgray", zorder=0)
        ax.plot([0.5, 0.9], [y, 0.5], color="lightgray", zorder=0)
    ax.scatter([0.1], [0.5], s=2800, color="#E0E0E0"); ax.text(0.1, 0.5, "tempo", ha="center", va="center")
    ax.scatter([0.5] * len(ys), ys, s=2800, color="#00A868")
    for i, y in enumerate(ys, start=1):
        ax.text(0.5, y, f"neuronio {i}", ha="center", va="center", fontsize=8, color="white")
    ax.scatter([0.9], [0.5], s=2800, color="#0B2E22"); ax.text(0.9, 0.5, "nota", ha="center", va="center", color="white")
    ax.text(0.1, 0.02, "entrada", ha="center")
    ax.text(0.5, 0.02, "camada oculta (as rampas)", ha="center")
    ax.text(0.9, 0.02, "saida (soma)", ha="center")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    plt.show()

desenhar_rede(3)

**Correspondência entre o que foi construído e a terminologia**

| O que foi construído | Terminologia |
|---|---|
| Uma rampa (reta + dobra) | **neurônio** |
| A "dobra" (valores negativos viram zero) | **função de ativação** (ReLU) |
| Os números de cada rampa | **pesos** (`w`, `b`, `v`, `c`) |
| O grupo de rampas lado a lado | **camada oculta** |
| A soma das rampas que produz a nota | **camada de saída** |
| O computador tentando e mantendo o que reduz o erro | **treinamento** |
| A escolha de quantas rampas usar | **hiperparâmetro** (número de neurônios) |

## Passo 8 — Implementação com o Scikit-Learn

O `MLPRegressor` (Multi-Layer Perceptron) implementa exatamente essa rede: neurônios com ReLU seguidos de uma soma final. O usuário escolhe apenas **quantos neurônios** a camada oculta terá (`hidden_layer_sizes`). Em vez de alterar os valores aleatoriamente, o `MLPRegressor` calcula para que lado cada valor deve mudar (o gradiente descendente, já estudado).

Na célula abaixo:

1. `StandardScaler` normaliza a entrada (procedimento já utilizado com outros algoritmos);
2. `MLPRegressor(...)` cria a rede. **COMPLETE** a quantidade de neurônios da camada oculta (3);
3. `fit` treina a rede e `predict` calcula as previsões;
4. `coefs_` e `intercepts_` armazenam os valores aprendidos, cuja quantidade é impressa.

Observação: neste notebook não há separação entre treino e teste, pois o objetivo é compreender a forma da curva.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

escala = StandardScaler().fit(X)
X_norm = escala.transform(X)

# COMPLETE: rede com 3 neuronios na camada oculta
rede = MLPRegressor(hidden_layer_sizes=(3,), activation="relu", max_iter=20000, random_state=0)
rede.fit(X_norm, nota)

grade = np.linspace(0, 60, 300).reshape(-1, 1)
previsao_rede = rede.predict(escala.transform(grade))
erro_rede = np.mean((rede.predict(X_norm) - nota) ** 2)

plt.scatter(tempo, nota, color="lightgray", s=18)
plt.plot(grade, previsao_rede, color="#00A868", linewidth=3)
plt.title(f"MLPRegressor {rede.hidden_layer_sizes} | erro (MSE) = {erro_rede:.2f}")
plt.xlabel("minutos no forno"); plt.ylabel("nota do bolo")
plt.show()

n_numeros = sum(w.size for w in rede.coefs_) + sum(b.size for b in rede.intercepts_)
print("Numeros que a rede aprendeu sozinha:", n_numeros)

**Interpretação.** A curva da rede acompanha a montanha, com MSE próximo de 0,7. A rede aprendeu 10 valores: 3 pesos de entrada (`w`), 3 vieses (`b`), 3 pesos de saída (`v`) e 1 viés de saída (`c`), exatamente os parâmetros da expressão apresentada no Passo 7.

A célula seguinte treina a mesma rede com 10 sementes diferentes (`random_state`) e imprime o MSE de cada uma.

In [ ]:
erros = []
for s in range(10):
    r = MLPRegressor(hidden_layer_sizes=(3,), activation="relu", max_iter=20000, random_state=s).fit(X_norm, nota)
    erros.append(round(float(np.mean((r.predict(X_norm) - nota) ** 2)), 2))
print(erros)

**Interpretação.** Nove das dez execuções apresentam MSE entre aproximadamente 0,6 e 1,4; uma permanece em aproximadamente 9,7. Assim como na busca do Passo 6, a rede começa com valores aleatórios e pode não convergir dependendo do ponto de partida. É por esse motivo que o parâmetro `random_state` é fixado ao treinar redes neurais.

## Passo 9 — Síntese

1. Havia um problema que a reta não resolvia: a nota precisava subir e depois voltar a descer.
2. A Regressão Polinomial resolve o problema dentro da faixa dos dados, mas diverge fora dela e escala mal com o número de entradas.
3. A alternativa é **emendar rampas**. Cada rampa é um **neurônio**; as rampas lado a lado formam a **camada oculta**.
4. Os números das rampas são os **pesos**. Eles não são escolhidos manualmente: o computador os **treina**, tentando alterações e mantendo as que reduzem o erro.
5. Para prever um número (regressão), a saída é a soma direta dos neurônios. Para prever uma categoria (classificação), apenas a **última camada** muda: a soma passa pela função sigmoide e é convertida em probabilidade, como na Regressão Logística.

**Próxima etapa:** implementação da mesma rede em Keras.